# Notebook 00 — Pre-flight

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

How do I know my Workshop 1 environment is complete enough for Workshop 2 to start?

Workshop 2 builds governed banking applications on top of the ontology, graph
database, and SHACL (Shapes Constraint Language) shapes that Workshop 1 produces.
If any piece of that foundation is missing, Workshop 2's agents will return empty
results, its verification cells will fail opaquely, and you will spend time
debugging the wrong layer. This notebook answers the question by checking the
foundation directly — before any Workshop 2 code runs.

This notebook reads from your environment. It never writes to it.

In [ ]:
# WS2 notebook setup — installs only what this notebook needs into the kernel.
# Skips uv sync (which installs the full agent stack and takes minutes).
import sys, subprocess

# Only install packages not already provided by the SageMaker base image
pkgs = ['rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0']

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check'] + pkgs,
    cwd='/tmp'
)
print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **Data contract** | The formal agreement between Workshop 1 and Workshop 2, defined in `spec/03-data-contracts.md`. Every assertion in this notebook is derived from that document. |
| **atlas: namespace** | The IRI (Internationalised Resource Identifier) prefix `https://github.com/your-org/atlas/ontology#` that all Workshop 1 ontology classes use. Workshop 2 references classes by their canonical `atlas:ClassName` local names. |
| **SLGD (Semantic Layer Graph Database)** | The curated Neptune cluster that holds the FIBO (Financial Industry Business Ontology)-aligned, SHACL-validated ontology and instance data. Workshop 2 agents query the SLGD. |
| **SHACL NodeShape** | A constraint rule in the Shapes Constraint Language (SHACL). Workshop 1 defines six NodeShapes that enforce the deterministic-vs-probabilistic boundary Workshop 2 depends on. |
| **CloudFormation stack output** | A named value exported from an AWS CloudFormation stack. Workshop 1's Neptune stack (`atlas-neptune-twotier`) exports the SLGD and LGD (Lexical Graph Database) endpoint hostnames as stack outputs. This notebook reads those values so you never paste an endpoint URL manually. |

## What Workshop 2 inherits from Workshop 1

Workshop 1 produces three categories of artifacts that Workshop 2 cannot function
without. The first is an ontology: 22 classes in the `atlas:` namespace, aligned
to FIBO via `rdfs:subClassOf` bindings, and loaded into a running Amazon Neptune
cluster. These classes are the vocabulary that every Workshop 2 agent speaks. When
the `nl-to-sparql-agent` translates a banker's question into a graph query, it
produces SPARQL (SPARQL Protocol and RDF Query Language) that references
`atlas:Customer`, `atlas:WealthSignal`, and `atlas:AdvisoryRelationship`. If those
classes are absent from the graph, the query returns nothing — and the agent has
no way to distinguish "this customer has no signals" from "the graph is empty."

The second category is six SHACL NodeShapes. These are the enforcement mechanism
that makes ATLAS compliant with SR 11-7 and OCC 2011-12 — the U.S. federal model
risk guidance that governs how AI-produced outputs may be used in banking decisions.
Each shape is a machine-readable rule: `atlas:BoundaryShape` asserts that every
Bedrock-drafted output carries a probabilistic flag before being shown to a user;
`atlas:ComplianceInputShape` asserts that every compliance-bound decision carries
the explainability artifacts the regulation requires. Workshop 2 does not add new
shapes — it relies entirely on Workshop 1's six. If any are missing, SHACL
validation cells in later notebooks will pass silently against an unconstrained
graph, which is exactly the failure mode the shapes exist to prevent.

The third category is instance data: 200 synthetic customers, 3,747 transactions,
10 advisors, and 105 advisory relationships. This is the corpus Workshop 2's agents
run against. Workshop 2 cannot generate meaningful wealth signals from an empty
graph, and it cannot demonstrate the referral orchestrator routing a customer to an
advisor if no advisors exist. The counts are fixed by a reproducible random seed so
that verification cells can assert specific numbers — a deterministic test is more
informative than an open-ended "looks reasonable" check.

The pre-flight notebook exists because the consequences of building on an incomplete
foundation are not obvious until much later. A missing class produces a SPARQL query
that returns zero rows — which looks identical to a correct query against a customer
who genuinely has no signals. A missing SHACL shape produces validation cells that
pass — which looks identical to validation against clean data. The errors are silent.
This notebook makes them loud, before Workshop 2 starts rather than after it is
half-built.

This notebook does not build artifacts; it verifies them.

## What we are about to verify

The six checks below map directly to the assertions in `spec/03-data-contracts.md`.
Each check prints what it found before it asserts, so you can read the output and
understand what the graph contains — not just whether a binary pass or fail fired.

The checks are: (1) Neptune connectivity, (2) ontology class count and required
class names, (3) SHACL shape names, (4) instance data counts, (5) Workshop 1 file
paths on disk, and (6) Bedrock model access. If any check fails, the cell raises
with a plain-English remediation message pointing to the Workshop 1 module that
produced the missing artifact.

In [ ]:
import json
import sys
import os

# Workshop 1's shared helpers live in agentic-semantic-layer/notebooks/shared/.
# From this notebook's location (use-case-applications/notebooks/phase-1-referral/),
# three levels up reaches the repo root, then down into the shared directory.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

import boto3
from pathlib import Path

from atlas_neptune import NeptuneClient
from atlas_sparql import build_prefixes

# The CloudFormation stack name that Workshop 1 deploys Neptune from.
STACK_NAME = "atlas-neptune-twotier"

# Repo root — used for file-path assertions in Check 5.
# Path(__file__) works when the notebook is executed as a script;
# the fallback resolves relative to the current working directory in Jupyter.
try:
    REPO_ROOT = Path(__file__).resolve().parents[3]
except NameError:
    REPO_ROOT = Path("../../..").resolve()

# AWS region — reads from environment or defaults to us-east-1.
AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

print(f"Shared helpers loaded from:  {sys.path[0]}")
print(f"Repo root resolved to:       {REPO_ROOT}")
print(f"CloudFormation stack name:   {STACK_NAME}")
print(f"AWS region:                  {AWS_REGION}")

In [ ]:
# Retrieve Neptune endpoints from the CloudFormation stack Workshop 1 deployed.
# This is the same pattern Workshop 1's Module 3 uses — the stack outputs are
# the single source of truth for endpoint addresses. No pasting URLs.

cfn = boto3.client("cloudformation", region_name=AWS_REGION)

print(f"Reading stack outputs from: {STACK_NAME}")
print("(If this raises, the stack does not exist or has not completed.)")
print()

try:
    response = cfn.describe_stacks(StackName=STACK_NAME)
    stack = response["Stacks"][0]
    stack_status = stack["StackStatus"]
    outputs = {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}
except cfn.exceptions.ClientError as exc:
    print(f"Stack '{STACK_NAME}' not found: {exc}")
    print()
    print("Remediation: Deploy the Neptune stack from Workshop 1 before running")
    print("  this notebook:")
    print("  agentic-semantic-layer/infrastructure/atlas-neptune-twotier.yaml")
    raise

SLGD_ENDPOINT = outputs.get("SLGDEndpoint", "")
LGD_ENDPOINT  = outputs.get("LGDEndpoint",  "")
SLGD_PORT     = int(outputs.get("SLGDPort", 8182))
LGD_PORT      = int(outputs.get("LGDPort",  8182))

print(f"Stack status:  {stack_status}")
print(f"SLGD endpoint: {SLGD_ENDPOINT}:{SLGD_PORT}")
print(f"LGD endpoint:  {LGD_ENDPOINT}:{LGD_PORT}")
print()

slgd = NeptuneClient(endpoint=SLGD_ENDPOINT, port=SLGD_PORT)
lgd  = NeptuneClient(endpoint=LGD_ENDPOINT,  port=LGD_PORT)

print("NeptuneClient instances ready: slgd, lgd")

In [ ]:
# Check 1 — Neptune connectivity
# The simplest possible query: count all triples in each cluster.
# A reachable, populated SLGD returns a positive number.
# A reachable but empty SLGD returns 0 — different problem, specific remediation.
# An unreachable cluster raises a requests exception.

print("Check 1 — Neptune connectivity")
print("-" * 50)

try:
    slgd_count = slgd.count_triples()
    print(f"  SLGD total triples: {slgd_count}")
except Exception as exc:
    print(f"  [FAIL] SLGD unreachable: {exc}")
    print()
    print("  Remediation: Verify the Neptune cluster is running and this SageMaker")
    print("  instance has network access to it (same VPC or VPC peering, port 8182).")
    print("  Redeploy from agentic-semantic-layer/infrastructure/atlas-neptune-twotier.yaml")
    print("  if the cluster was deleted to save cost.")
    raise

try:
    lgd_count = lgd.count_triples()
    print(f"  LGD total triples:  {lgd_count}")
except Exception as exc:
    print(f"  [FAIL] LGD unreachable: {exc}")
    print()
    print("  Remediation: Same as SLGD — check VPC routing and cluster status.")
    raise

assert slgd_count > 0, (
    f"SLGD is reachable but empty ({slgd_count} triples). "
    "Re-run Workshop 1 modules 3 and 4 to reload the ontology and synthetic data."
)

print()
print("  [PASS] Check 1 — both clusters reachable, SLGD populated")
CHECK_1 = True

In [ ]:
# Check 2 — Ontology class count and required class names
# The data contract requires at least 22 atlas: classes. We assert >= 22,
# not exactly 22, so that a novice who extended the ontology with their own
# classes does not fail this gate — the contract is a floor, not a ceiling.
# We then check each of the 15 named required classes individually, because
# those are the class names Workshop 2's agents call by name.

print("Check 2 — Ontology class count and required classes")
print("-" * 50)

ATLAS_IRI = "https://github.com/your-org/atlas/ontology#"

q_classes = (
    build_prefixes()
    + f"""
SELECT ?cls WHERE {{
    ?cls a owl:Class .
    FILTER(STRSTARTS(STR(?cls), "{ATLAS_IRI}"))
}}
ORDER BY ?cls
"""
)

rows = slgd.query(q_classes)
found_classes = {r["cls"].split("#")[-1] for r in rows}

print(f"  atlas: classes found ({len(found_classes)}):")
for cls_name in sorted(found_classes):
    print(f"    atlas:{cls_name}")
print()

assert len(found_classes) >= 22, (
    f"Expected >= 22 atlas: classes, found {len(found_classes)}. "
    "Re-run Workshop 1 modules 1 and 2 to regenerate and reload the ontology."
)

# These 15 classes are the ones Workshop 2 agents reference by name.
# The remaining 7 (to reach 22) are also required but not called by name in Phase 1.
REQUIRED_CLASSES = [
    "Customer", "Account", "Holding", "Transaction", "Household",
    "WealthSignal", "WealthSignalType", "Advisor", "AdvisoryRelationship",
    "RoutingDecision", "HumanReview", "AuditRecord",
    "LegalEntity", "Product", "LineOfBusiness",
]

missing = [c for c in REQUIRED_CLASSES if c not in found_classes]
if missing:
    print(f"  [FAIL] Missing required classes: {['atlas:' + c for c in missing]}")
    print()
    print("  Remediation: The listed classes must be present in atlas-core.ttl or")
    print("  atlas-fibo-alignment.ttl. Re-run Workshop 1 module 3 to reload the ontology.")
    raise AssertionError(f"Required Workshop 1 classes missing from SLGD: {missing}")

print(f"  [PASS] Check 2 — {len(found_classes)} classes found, all 15 required classes present")
CHECK_2 = True

In [ ]:
# Check 3 — SHACL shapes
# Workshop 1 defines six NodeShapes in atlas-shapes.ttl. Workshop 2's
# boundary enforcement depends on all six being present and queryable in
# the SLGD. This is the same query Workshop 1's Module 6 validation cell runs.

print("Check 3 — SHACL shapes")
print("-" * 50)

q_shapes = (
    build_prefixes()
    + f"""
PREFIX sh: <http://www.w3.org/ns/shacl#>
SELECT ?shape WHERE {{
    ?shape a sh:NodeShape .
    FILTER(STRSTARTS(STR(?shape), "{ATLAS_IRI}"))
}}
ORDER BY ?shape
"""
)

rows = slgd.query(q_shapes)
found_shapes = {r["shape"].split("#")[-1] for r in rows}

print(f"  SHACL NodeShapes found ({len(found_shapes)}):")
for shape_name in sorted(found_shapes):
    print(f"    atlas:{shape_name}")
print()

REQUIRED_SHAPES = [
    "ProvenanceShape", "BoundaryShape", "ComplianceInputShape",
    "RoutingPolicyShape", "WealthSignalTypeShape", "CoverageRelationshipShape",
]

missing_shapes = [s for s in REQUIRED_SHAPES if s not in found_shapes]
if missing_shapes:
    print(f"  [FAIL] Missing required shapes: {['atlas:' + s for s in missing_shapes]}")
    print()
    print("  Remediation: Re-run Workshop 1 module 6. The six SHACL shapes must be")
    print("  loaded from agentic-semantic-layer/ontology/atlas-shapes.ttl into the SLGD.")
    raise AssertionError(f"Required Workshop 1 SHACL shapes missing from SLGD: {missing_shapes}")

print(f"  [PASS] Check 3 — {len(found_shapes)} shapes found, all 6 required shapes present")
CHECK_3 = True

In [ ]:
# Check 4 — Instance data counts
# Workshop 1's synthetic data is the corpus Workshop 2 agents run against.
# All checks use >= floors: Transaction may be higher if nb05 was re-run,
# and Advisor has exactly 9 unique IDs in the synthetic corpus.
# The most common cause of a low count is that module 4 was interrupted.

print("Check 4 — Instance data counts")
print("-" * 50)

EXPECTED_COUNTS = {
    "Customer":             200,
    # nb04 caps transaction promotion at 500 for workshop speed;
    # nb05 promotes those 500 to the SLGD. Full corpus = 3747.
    "Transaction":          500,  # floor — may be higher if nb05 re-ran
    "Advisor":              9,   # synthetic data has 9 unique advisor IDs
    "AdvisoryRelationship": 105,
}

all_counts_ok = True
for cls_name, expected in EXPECTED_COUNTS.items():
    q = (
        build_prefixes()
        + f"SELECT (COUNT(?inst) AS ?n) WHERE {{ ?inst a atlas:{cls_name} }}"
    )
    rows = slgd.query(q)
    actual = int(rows[0]["n"]) if rows else 0
    status = "[PASS]" if actual >= expected else "[FAIL]"
    print(f"  {status} atlas:{cls_name:<25} expected >={expected:>4}   found {actual:>5}")
    if actual < expected:
        all_counts_ok = False

print()

if not all_counts_ok:
    print("  Remediation: Re-run Workshop 1 modules 4 and 5 to reload the synthetic data.")
    print("  Transaction count reflects the first 500 transactions (workshop cap).")
    print("  If you regenerated the synthetic data with a different random seed,")
    print("  the counts will differ. Use the original generator with seed=42.")
    raise AssertionError(
        "Instance data counts do not match the Workshop 1 data contract. "
        "See output above for which counts failed."
    )

print("  [PASS] Check 4 — all instance data counts match the data contract")
CHECK_4 = True

In [ ]:
# Check 5 — Required Workshop 1 file paths
# Workshop 2's agents and notebooks read four Workshop 1 files directly at
# runtime. If any are absent the agent fails with a FileNotFoundError at the
# worst possible moment — mid-notebook, mid-demo. Better to catch it here.

print("Check 5 — Required Workshop 1 file paths")
print("-" * 50)

REQUIRED_FILES = [
    "agentic-semantic-layer/prompts/prefixes.txt",
    "agentic-semantic-layer/prompts/ground-truth.yaml",
    "agentic-semantic-layer/notebooks/shared/atlas_neptune.py",
    "agentic-semantic-layer/notebooks/shared/atlas_sparql.py",
]

all_files_ok = True
for rel_path in REQUIRED_FILES:
    full_path = REPO_ROOT / rel_path
    exists = full_path.exists()
    status = "[PASS]" if exists else "[FAIL]"
    print(f"  {status} {rel_path}")
    if not exists:
        all_files_ok = False

print()

if not all_files_ok:
    print("  Remediation: Verify that your git checkout includes the full")
    print("  agentic-semantic-layer/ directory. Run:")
    print("    git checkout main -- agentic-semantic-layer/")
    print("  If the file is genuinely absent from the repo, re-run Workshop 1.")
    raise AssertionError(
        "One or more required Workshop 1 files are missing from disk. "
        "See output above for which files failed."
    )

print("  [PASS] Check 5 — all required Workshop 1 files present on disk")
CHECK_5 = True

In [ ]:
# Check 6 — Bedrock model access
# Workshop 2 uses two Bedrock foundation models.
#   nl-to-sparql-agent uses Amazon Titan Embeddings v2 for embedding-based
#   template selection — this is the lookup, not generation, which is why it
#   is an embedding model rather than a text model.
#   referral-rationale-drafter uses Claude Sonnet on Bedrock for narrative
#   drafting (the probabilistic-edge output that always requires human approval).
#
# This check does a live invoke rather than listing APIs. Listing tells you a
# model exists; invoking tells you the role can actually call it. Workshop 2
# agents need bedrock:InvokeModel — that is the permission that matters here.

print("Check 6 — Bedrock model access")
print("-" * 50)

bedrock_rt = boto3.client("bedrock-runtime", region_name=AWS_REGION)

REQUIRED_MODELS = {
    "us.anthropic.claude-sonnet-4-6": {
        "used_by": "referral-rationale-drafter (narrative drafting)",
        "body": json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1,
            "messages": [{"role": "user", "content": "hi"}],
        }),
        "content_type": "application/json",
        "accept": "application/json",
    },
    "amazon.titan-embed-text-v2:0": {
        "used_by": "nl-to-sparql-agent (embedding-based template selection)",
        "body": json.dumps({"inputText": "test"}),
        "content_type": "application/json",
        "accept": "application/json",
    },
}

all_models_ok = True
for model_id, cfg in REQUIRED_MODELS.items():
    try:
        bedrock_rt.invoke_model(
            modelId=model_id,
            body=cfg["body"],
            contentType=cfg["content_type"],
            accept=cfg["accept"],
        )
        print(f"  [PASS] {model_id}")
        print(f"         Used by: {cfg['used_by']}")
    except bedrock_rt.exceptions.AccessDeniedException as exc:
        print(f"  [FAIL] {model_id}")
        print(f"         Used by: {cfg['used_by']}")
        print(f"         Error: {exc}")
        all_models_ok = False
    except Exception as exc:
        # Any non-access error (throttle, validation) still means the role CAN invoke —
        # the request reached Bedrock. Only AccessDeniedException means it cannot.
        print(f"  [PASS] {model_id}  (invoke reached Bedrock; non-access error: {type(exc).__name__})")
        print(f"         Used by: {cfg['used_by']}")

print()

if not all_models_ok:
    print("  Remediation: Open the AWS Bedrock console → Model access and enable the")
    print("  listed models. Anthropic models may require a use-case acknowledgement.")
    print("  Workshop 1 module 7 walks through this step — if you completed it,")
    print("  model access should already be enabled.")
    print()
    print("  If the error is an IAM AccessDeniedException (not model access), the")
    print("  SageMaker execution role needs bedrock:InvokeModel on these model ARNs.")
    raise AssertionError(
        "One or more required Bedrock models are not accessible in this account/region. "
        "See output above for which models failed."
    )

print("  [PASS] Check 6 — all required Bedrock models accessible")
CHECK_6 = True


In [ ]:
# Pre-flight validation gate — summary across all six checks.
# If any earlier cell raised, its CHECK_N variable was never set.
# locals().get() with a False default surfaces those as failures here.

print("=" * 60)
print("PRE-FLIGHT VALIDATION GATE")
print("=" * 60)
print()

checks = {
    "Check 1 — Neptune connectivity":              locals().get("CHECK_1", False),
    "Check 2 — Ontology classes":                  locals().get("CHECK_2", False),
    "Check 3 — SHACL shapes":                      locals().get("CHECK_3", False),
    "Check 4 — Instance data counts":              locals().get("CHECK_4", False),
    "Check 5 — Required file paths":               locals().get("CHECK_5", False),
    "Check 6 — Bedrock model access":              locals().get("CHECK_6", False),
}

all_passed = True
for label, passed in checks.items():
    marker = "PASS" if passed else "FAIL / NOT RUN"
    print(f"  {marker:<16} {label}")
    if not passed:
        all_passed = False

print()

if all_passed:
    print("PRE-FLIGHT: PASS")
    print("Workshop 1's substrate is confirmed. Workshop 2 is safe to start.")
    print("Open notebook 01_why_agents.ipynb to continue.")
else:
    print("PRE-FLIGHT: FAIL")
    print("One or more checks did not pass. Scroll up to the first FAIL cell.")
    print("Each failed cell prints the specific remediation for that check.")
    print("Fix the failures and re-run from the top before opening any other")
    print("Workshop 2 notebook.")
    raise AssertionError("Pre-flight failed. See check output above for remediation steps.")

## Persisting Workshop 1 outputs for the Workshop 2 CDK deploy

Workshop 2 deploys into **your** AWS account. The CDK stack that creates the
Agents, MCP servers, GraphQL API, and UIs needs to know where your Workshop 1
infrastructure lives — the Neptune endpoints, the VPC, the subnets, and the
S3 staging bucket. These values are different in every runner's account.

The cell below reads those values from your Workshop 1 CloudFormation stack
(the same `atlas-neptune-twotier` stack the checks above just verified) and
writes them into `use-case-applications/cdk/cdk.json`. This is the bridge
between the two workshops: instead of you pasting endpoint addresses manually
before running `cdk deploy`, the bridge discovers them automatically and
persists them exactly once.

Two values — the VPC ID and the private subnet IDs — are not direct
CloudFormation outputs, so the cell derives them from the Neptune security
group ID (which is a stack output) via read-only EC2 API calls.

After this cell runs, `cdk synth` and `cdk deploy` in
`use-case-applications/cdk/` will use your Workshop 1 values with no further
configuration.

In [ ]:
# WS1 → WS2 bridge — discover Workshop 1 outputs and persist CDK deploy context.
#
# WHY THIS CELL EXISTS: Workshop 2's CDK stack deploys agents, MCP servers, and
# UIs into YOUR account. It needs your Workshop 1 Neptune endpoints, VPC, subnets,
# and S3 staging bucket. These differ in every runner's account and must NEVER be
# committed to the repo. This cell discovers them once, from your live Workshop 1
# deployment, and writes them to cdk.json so cdk deploy works without manual config.
#
# Five context keys the WS2 CDK stack reads (via cdk.json or --context flags):
#   neptuneClusterEndpoint  — SLGD cluster endpoint hostname
#   neptuneLgdEndpoint      — LGD cluster endpoint hostname
#   ontologyStagingBucket   — S3 bucket for ontology + prompt artifacts
#   vpcId                   — VPC the WS2 stack deploys into (same as WS1)
#   privateSubnetIds        — comma-joined private subnet IDs for ECS/Lambda

print("WS1 → WS2 bridge: persisting deploy context to cdk.json")
print("-" * 60)

# ── Step 1: resolve the five values ───────────────────────────────────────
# Three come directly from the WS1 CFN outputs (already in `outputs` from cell-06):
slgd_endpoint    = outputs.get("SLGDEndpoint", "")
lgd_endpoint     = outputs.get("LGDEndpoint", "")
staging_bucket   = outputs.get("OntologyStagingBucketName", "")
neptune_sg_id    = outputs.get("NeptuneSecurityGroupId", "")

# Safety check — all three must be present before we proceed
missing = [k for k, v in {
    "SLGDEndpoint": slgd_endpoint,
    "LGDEndpoint": lgd_endpoint,
    "OntologyStagingBucketName": staging_bucket,
    "NeptuneSecurityGroupId": neptune_sg_id,
}.items() if not v]
if missing:
    print(f"  [FAIL] Missing CFN outputs: {missing}")
    print()
    print("  Remediation: The listed outputs are missing from the atlas-neptune-twotier")
    print("  stack. Re-deploy Workshop 1 and re-run this notebook from the top.")
    raise AssertionError(f"Required WS1 CFN outputs missing: {missing}")

print(f"  SLGD endpoint:      {slgd_endpoint}")
print(f"  LGD endpoint:       {lgd_endpoint}")
print(f"  Staging bucket:     {staging_bucket}")
print(f"  Neptune SG:         {neptune_sg_id}")

# Two more come from the Neptune security group (not a direct CFN output):
# VPC ID ← describe the Neptune SG → VpcId
# Private subnets ← describe subnets in that VPC whose Name contains "Private"
ec2 = boto3.client("ec2", region_name=AWS_REGION)

sg_resp = ec2.describe_security_groups(GroupIds=[neptune_sg_id])
vpc_id = sg_resp["SecurityGroups"][0]["VpcId"]
print(f"  VPC ID (derived):   {vpc_id}")

subnets_resp = ec2.describe_subnets(
    Filters=[
        {"Name": "vpc-id", "Values": [vpc_id]},
    ]
)
# Select private subnets — those whose Name tag contains "Private"
private_subnets = [
    s["SubnetId"] for s in subnets_resp["Subnets"]
    if any(
        "Private" in t.get("Value", "")
        for t in s.get("Tags", [])
        if t["Key"] == "Name"
    )
]
if not private_subnets:
    # Fallback: subnets with no auto-assign public IP
    private_subnets = [
        s["SubnetId"] for s in subnets_resp["Subnets"]
        if not s.get("MapPublicIpOnLaunch", True)
    ]
if not private_subnets:
    print("  [FAIL] No private subnets found in VPC.")
    print()
    print("  Remediation: Verify the VPC has private subnets (subnets whose Name tag")
    print("  contains 'Private' or whose auto-assign public IP is disabled). The")
    print("  Workshop 1 VPC is the SageMaker Unified Studio domain VPC and should have")
    print("  at least four private subnets.")
    raise AssertionError("No private subnets found — cannot populate privateSubnetIds for CDK.")
private_subnet_ids_str = ",".join(sorted(private_subnets))
print(f"  Private subnets:    {private_subnet_ids_str}")

# ── Step 2: write to cdk.json ──────────────────────────────────────────────
# Path from this notebook to cdk.json:
#   use-case-applications/notebooks/phase-1-referral/  ← this notebook
#   use-case-applications/cdk/cdk.json                 ← target
cdk_json_path = Path("../../cdk/cdk.json").resolve()
if not cdk_json_path.exists():
    raise FileNotFoundError(
        f"cdk.json not found at {cdk_json_path}. "
        "Verify the notebook is run from use-case-applications/notebooks/phase-1-referral/."
    )

cdk_config = json.loads(cdk_json_path.read_text())
cdk_config["context"].update({
    "neptuneClusterEndpoint": slgd_endpoint,
    "neptuneLgdEndpoint":     lgd_endpoint,
    "ontologyStagingBucket":  staging_bucket,
    "vpcId":                  vpc_id,
    "privateSubnetIds":       private_subnet_ids_str,
})
cdk_json_path.write_text(json.dumps(cdk_config, indent=2))

print()
print(f"  Written to: {cdk_json_path}")
print()
print("  [PASS] WS1 → WS2 bridge complete. Run `cdk deploy` from use-case-applications/cdk/")
print("         when you are ready to deploy Workshop 2.")


## Uploading Workshop 2 artifacts to your staging bucket

Workshop 2's five agents each read one or more files from S3 at cold-start:

| Agent / MCP server | Environment variable | S3 key |
|---|---|---|
| atlas-shacl-mcp | `SHAPES_S3_URI` | `ontology/atlas-shapes.ttl` |
| nl-to-sparql-agent | `GROUND_TRUTH_S3_URI` | `prompts/ground-truth.yaml` |
| nl-to-sparql-agent | `PREFIXES_S3_URI` | `prompts/prefixes.txt` |
| referral-rationale-drafter | `PROMPT_TEMPLATE_S3_URI` | `prompts/referral-rationale.txt` |
| wealth-signal-detector | `SIGNAL_QUERIES_S3_URI` | `queries/wealth-signals.yaml` |

Workshop 1 uploads `atlas-shapes.ttl` for its own bulk-load. The other four are Workshop 2-owned
artifacts. The cell below uploads all five to **your** staging bucket — the same bucket the bridge
cell just wrote into `cdk.json`. Run this cell once after the bridge cell. Re-running is safe
(uploads overwrite in place).


In [ ]:
# WS2 artifact upload — populate the runner's staging bucket with the five objects
# the WS2 agents read at runtime.
#
# WHY THIS CELL EXISTS: Workshop 2's agents resolve S3 URIs from environment variables
# that the CDK stack sets (e.g. SHAPES_S3_URI, GROUND_TRUTH_S3_URI). Those URIs all
# point to the runner's own staging bucket — the bucket Workshop 1 created and the
# bridge cell above just persisted as ontologyStagingBucket in cdk.json. This cell
# puts the actual artifact bytes there so the agents can read them on first invocation.
#
# Five objects, one upload per env var the agents depend on:
#
#   LOCAL SOURCE                                         → S3 KEY
#   agentic-semantic-layer/ontology/atlas-shapes.ttl     → ontology/atlas-shapes.ttl
#   agentic-semantic-layer/prompts/ground-truth.yaml     → prompts/ground-truth.yaml
#   agentic-semantic-layer/prompts/prefixes.txt          → prompts/prefixes.txt
#   use-case-applications/prompts/rationale-template.txt → prompts/referral-rationale.txt
#   use-case-applications/prompts/signal-queries/        → queries/wealth-signals.yaml
#       wealth-signals.yaml

print("WS2 artifact upload: populating staging bucket with agent runtime artifacts")
print("-" * 60)

# ── Step 1: resolve the staging bucket ────────────────────────────────────────
# The bridge cell above stored staging_bucket in scope. Re-read from cdk.json
# in case this cell is run independently (e.g. after a kernel restart).
if not staging_bucket:
    cdk_config = json.loads(cdk_json_path.read_text())
    staging_bucket = cdk_config["context"].get("ontologyStagingBucket", "")

if not staging_bucket:
    print("  [FAIL] ontologyStagingBucket is empty.")
    print()
    print("  Remediation: Run the bridge cell above (WS1 → WS2 bridge) first.")
    print("  That cell discovers your staging bucket from the atlas-neptune-twotier")
    print("  stack outputs and persists it to cdk.json.")
    raise AssertionError("ontologyStagingBucket is empty — run the bridge cell first.")

print(f"  Target bucket: s3://{staging_bucket}/")
print()

# ── Step 2: define the five upload mappings ────────────────────────────────────
# Paths are relative from this notebook's directory:
#   notebooks/phase-1-referral/ → up 3 levels → repo root / agentic-semantic-layer
#                                → up 2 levels → use-case-applications/prompts
NOTEBOOK_DIR = Path(__file__ if "__file__" in dir() else ".").resolve().parent
if str(NOTEBOOK_DIR) == ".":
    # When running as a Jupyter kernel, __file__ is not set; resolve from cwd
    NOTEBOOK_DIR = Path.cwd()

# Walk up to the repo root (two levels above use-case-applications/)
REPO_ROOT = NOTEBOOK_DIR.parent.parent.parent

ARTIFACTS = [
    {
        "src": REPO_ROOT / "agentic-semantic-layer" / "ontology" / "atlas-shapes.ttl",
        "key": "ontology/atlas-shapes.ttl",
        "env_var": "SHAPES_S3_URI",
    },
    {
        "src": REPO_ROOT / "agentic-semantic-layer" / "prompts" / "ground-truth.yaml",
        "key": "prompts/ground-truth.yaml",
        "env_var": "GROUND_TRUTH_S3_URI",
    },
    {
        "src": REPO_ROOT / "agentic-semantic-layer" / "prompts" / "prefixes.txt",
        "key": "prompts/prefixes.txt",
        "env_var": "PREFIXES_S3_URI",
    },
    {
        "src": REPO_ROOT / "use-case-applications" / "prompts" / "rationale-template.txt",
        "key": "prompts/referral-rationale.txt",
        "env_var": "PROMPT_TEMPLATE_S3_URI",
    },
    {
        "src": REPO_ROOT / "use-case-applications" / "prompts" / "signal-queries" / "wealth-signals.yaml",
        "key": "queries/wealth-signals.yaml",
        "env_var": "SIGNAL_QUERIES_S3_URI",
    },
]

# ── Step 3: fail-fast on any missing local source ─────────────────────────────
missing_sources = [a for a in ARTIFACTS if not a["src"].exists()]
if missing_sources:
    for a in missing_sources:
        print(f"  [FAIL] Source not found: {a['src']}")
    print()
    print("  Remediation: Ensure you are running this notebook from the")
    print("  use-case-applications/notebooks/phase-1-referral/ directory,")
    print("  and that both workshops are checked out at the expected paths.")
    raise FileNotFoundError(f"Missing source files: {[str(a['src']) for a in missing_sources]}")

# ── Step 4: upload ─────────────────────────────────────────────────────────────
s3_upload = boto3.client("s3", region_name=AWS_REGION)

for artifact in ARTIFACTS:
    s3_upload.upload_file(str(artifact["src"]), staging_bucket, artifact["key"])
    print(f"  [OK] {artifact['env_var']}")
    print(f"       {artifact['src'].name} → s3://{staging_bucket}/{artifact['key']}")

print()
print(f"  [PASS] All 5 artifacts uploaded to s3://{staging_bucket}/")
print()
print("  The WS2 agents will read these objects on first invocation after cdk deploy.")
print("  Re-running this cell is safe — uploads overwrite in place.")

## What just changed

Workshop 1's substrate is confirmed in the state Workshop 2 expects: 22 or more
ontology classes are present and queryable in the SLGD, the six SHACL shapes that
enforce the deterministic boundary are loaded, the synthetic data corpus is
populated with the expected counts, the shared helper files are on disk, and the
Bedrock models Workshop 2's agents call are accessible in this account.

The next notebook — `01_why_agents.ipynb` — introduces the architectural pattern
that every Phase 1 agent implements: LLM (Large Language Model) at the edges,
deterministic reasoning in the middle. That pattern is what makes ATLAS auditable
by a regulator, and it is the most important idea in Workshop 2.